# Reading the learned pathway units, on a dataset whose answer is known

Kang et al. (2018) is PBMCs under interferon-beta against control. The answer
is fixed before anything runs: a model that uses its annotation should name
`REACTOME_INTERFERON_ALPHA_BETA_SIGNALING`. This notebook fits **one** model,
under the architecture `01_architecture.py` selected, and reads the condition
effect straight off the model's own units.

**What changed, and why there is only one notebook now.** The previous pair —
`01_kang_end_to_end.ipynb` and `01_kang_showcase.ipynb` — reported a
gene-level competitive test as the model's headline result. That test ranks
this pathway **first in every arm of `01_architecture.py`, including the ones
whose gene→pathway map is shuffled**. It never sees the annotation, so it
could not have been reading the model's units. Ranking the units themselves,
the same pathway sat at **1,615 of 1,615** under the shipped architecture.

Three flags fix that, and the decision script shows each is necessary:

| flag | what it does |
| --- | --- |
| `cov_mode="pathway"` | the covariate's effect is a vector in pathway coordinates, so it reaches a gene only through the mask |
| `cov_in_encoder=False` | and it is the *whole* conditional effect, because the covariate cannot also reach `z` |
| `nonneg_decoder=True` | and a unit rising raises its genes, so the sign of a pathway means what it says |

Under all three the condition effect **is** one signed number per unit,
`InformedVAE.pathway_contrast`, with no cells, no sampling and no
counterfactual convention to choose.

| | |
| --- | --- |
| **Reads** | the Kang counts (downloaded by `pyvae.load_kang`), `resources/*.gmt`, `outputs/architecture/summary.json` |
| **Writes** | `outputs/units/` — `unit_contrasts.csv`, `ig_target.csv`, `target_cells.csv`, `summary.json` |
| **Figures** | `05_figures.py`, from those tables, on a CPU in seconds |
| **Run by** | `snakemake units`, one GPU, ~5 min |

## 1. Parameters

In [ ]:
# Parameters. papermill injects overrides in a cell directly below this one.
smoke = False
epochs = 250
seed = 42

In [ ]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import torch
from sklearn.model_selection import train_test_split

from pyvae import (
    InformedVAE,
    build_model_config,
    integrated_gradients,
    load_kang,
    set_all_seeds,
    train_ivae_modern,
    unit_sign_structure,
)

# SMOKE trades fidelity for speed so every cell can be exercised on a laptop. It
# is a syntax check with numbers attached; any figure produced under it should be
# thrown away, which is why it writes to its own directory.
SMOKE = bool(smoke) or os.environ.get("PYVAE_SMOKE", "0") == "1"

# Jupyter and papermill both start the kernel in the notebook's own directory.
# Normalise before touching the filesystem: `load_kang` downloads the dataset
# when it cannot find it, so a wrong working directory silently pulls 38 MB into
# experiments/kang/experiments/kang/data and the notebook appears to work.
_here = Path.cwd()
for _root in (_here, *_here.parents):
    if (_root / "pyvae" / "__init__.py").exists():
        os.chdir(_root)
        break
else:
    raise RuntimeError(f"could not find the pyvae repository root from {_here}")

SEED = int(seed)
DATA_FOLDER = "experiments/kang/data"
GMT = "experiments/kang/resources/c2.cp.reactome.v7.5.1.symbols.gmt"
OUT = Path("experiments/kang/outputs")
ARCH_SUMMARY = OUT / "architecture" / "summary.json"

#: The architecture `01_architecture.py` selected. The three flags after the
#: first four are the result of that script, not a preference; `BASE` alone is
#: the arm it beats.
BASE = dict(normalize="batch", informed_decoder=True, nonneg_encoder=True,
            standardize_input=True)
ARCH = dict(**BASE, cov_mode="pathway", cov_in_encoder=False,
            nonneg_decoder=True)
TRAIN = dict(epochs=int(epochs), patience=int(epochs), batch_size=128, lr=1e-3,
             weight_decay=1e-6, warmup_epochs=10, max_grad_norm=1.0)
N_GENES = None          # None means the full panel
TARGET_SUM = 1e4
L2_LAMBDA = 1e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if SMOKE:
    TRAIN |= dict(epochs=4, patience=4)
    N_GENES = 2000
    DEVICE = "cpu"

OUT_DIR = OUT / ("units_smoke" if SMOKE else "units")
OUT_DIR.mkdir(parents=True, exist_ok=True)

#: The answer, fixed before the experiment runs, and its Reactome parent.
#: Reactome nests, so the parent ranking alongside the child is agreement.
TARGET = "REACTOME_INTERFERON_ALPHA_BETA_SIGNALING"
PARENT = "REACTOME_INTERFERON_SIGNALING"
CTRL, STIM = "condition_control", "condition_stimulated"

# The figures are drawn by `05_figures.py`, which owns the palette and the
# style. This notebook writes tables.

set_all_seeds(SEED)
print(f"working directory  {Path.cwd()}")
print(f"torch {torch.__version__} | device {DEVICE} | smoke={SMOKE}")
print(f"writing to         {OUT_DIR}")

## 2. Data, the Reactome map, and the design

Nothing here changed: the same cells, the same map, the same two-block design
of condition and cell type. What changed is where the condition block enters
the model, which is section 3.

In [ ]:
adata = load_kang(data_folder=DATA_FOLDER, normalize=True, n_genes=N_GENES,
                  return_path=False, target_sum=TARGET_SUM)
x_trans = adata.to_df()
config = build_model_config(genes=x_trans.columns, model_kind="ivae_reactome",
                            resources_dir=GMT)
adj_df = config.model_layer[0]
gene_names = list(config.input_genes)
pathway_names = list(adj_df.columns)
x_trans = x_trans[gene_names]
counts_df = adata.to_df(layer="counts")[gene_names]

cov = pd.get_dummies(adata.obs[["condition", "cell_type"]], dtype=np.float32)
assert {CTRL, STIM} <= set(cov.columns), sorted(cov.columns)
n_cov = cov.shape[1]

strat = adata.obs["condition"].astype(str) + "_" + adata.obs["cell_type"].astype(str)
if strat.value_counts().min() < 2:
    strat = adata.obs["condition"].astype(str)
idx_tr, idx_va = train_test_split(np.arange(len(adata)), test_size=0.2,
                                  stratify=strat, random_state=SEED)
# A trailing batch of one row crashes BatchNorm; move that cell to validation.
if len(idx_tr) % TRAIN["batch_size"] == 1:
    idx_va, idx_tr = np.append(idx_va, idx_tr[-1]), idx_tr[:-1]

data = {"x_train": x_trans.iloc[idx_tr].astype(np.float32),
        "x_val": x_trans.iloc[idx_va].astype(np.float32),
        "counts_train": counts_df.iloc[idx_tr].astype(np.float32),
        "counts_val": counts_df.iloc[idx_va].astype(np.float32),
        "cov_train": cov.iloc[idx_tr], "cov_val": cov.iloc[idx_va]}

sizes = adj_df.sum(axis=0)
print(f"cells x genes      {adata.n_obs:,} x {adata.n_vars:,}")
print(f"pathway map        {len(gene_names):,} genes x {len(pathway_names):,} units"
      f"   (median {sizes.median():.0f} genes, max {sizes.max():.0f})")
print(f"design             {n_cov} columns: condition (2) x cell type "
      f"({adata.obs['cell_type'].nunique()})")
print(f"train / val        {len(idx_tr):,} / {len(idx_va):,} cells")
for p in (TARGET, PARENT):
    print(f"{'present' if p in pathway_names else 'ABSENT ':<8} {p} "
          f"({int(sizes[p]) if p in pathway_names else 0} genes)")

## 3. The model

`01_architecture.py` is the argument for this configuration and is not
re-argued here. The short form: the shipped decoder computed

```
logits = dec_out(tanh(dec_latent(z))) + cov_decoder(cov)
```

with `dec_out` masked by Reactome and `cov_decoder` a dense `(n_genes, n_cov)`
term added **after** it. Every condition contrast was that dense term and
nothing else, which is why shuffling the annotation changed nothing. Here the
covariate enters *inside* the pathway layer instead:

```
logits = dec_out(tanh(dec_latent(z) + cov_path(cov)))
```

and `cov_in_encoder=False` removes the second route, so `cov_path` is the whole
of it. `nonneg_decoder=True` reparameterises `dec_out` as `softplus(theta)`,
which does **not** mean the model cannot lower a gene — `tanh` is signed, so a
unit falling lowers every member — it means direction is carried by the unit
rather than scattered across its genes.

In [ ]:
latent_dim = len(pathway_names) // 2
set_all_seeds(SEED)
model = InformedVAE(adj=torch.tensor(adj_df.values, dtype=torch.float32),
                    latent_dim=latent_dim, seed=SEED, l2_lambda=L2_LAMBDA,
                    likelihood="nb", n_cov=n_cov, **ARCH)
print("architecture:", ARCH)
print(f"pathway units      {len(pathway_names):,}    latent dim {latent_dim:,}")
print(f"covariate term     cov_path {tuple(model.decoder.cov_path.weight.shape)}"
      f"   (n_pathways x n_cov), no gene-level term:"
      f" {not hasattr(model.decoder, 'cov_decoder')}")

t0 = time.time()
model, history = train_ivae_modern(
    model, x_train=data["x_train"], x_val=data["x_val"],
    x_counts_train=data["counts_train"], x_counts_val=data["counts_val"],
    cov_train=data["cov_train"], cov_val=data["cov_val"], device=DEVICE, **TRAIN)
train_min = (time.time() - t0) / 60
hist = pd.DataFrame(history)
DEV = next(model.parameters()).device
best_val = float(hist["val"].min())
print(f"\ntrained            {train_min:.1f} min, best val {best_val:.1f} "
      f"at epoch {int(hist['val'].idxmin()) + 1}")

## 4. Four checks, all pass/fail

Nothing below section 5 is readable unless all four hold, so they are
`assert`s rather than printed numbers to be eyeballed.

1. **No inverted encoder unit.** `nonneg_encoder` reparameterises every encoder
   weight as `softplus(theta) * mask`, so a negative live weight is
   unrepresentable. This is the one claim the retired fidelity grid established
   that still stands, and it must be exactly zero.
2. **No negative decoder weight.** The same for `dec_out`, and the reason a
   unit's sign transfers to its genes. On the shipped fit this was 0.50.
3. **Sign agreement is exactly 1.** Every unit's own direction matches the mean
   direction of its member genes. Measured, not assumed.
4. **`pathway_contrast` equals the forward pass.** The read-out reads a weight
   matrix; the model computes a forward pass. They must agree to float32
   precision or the read-out is describing a different model.

Saturation is reported rather than asserted: a unit stuck in the flat tail of
`tanh` carries no gradient, and `normalize="batch"` was chosen to keep it low.

In [ ]:
# The two covariate settings: the same cells, told they were control and told
# they were stimulated. Cell type is untouched, so this is the condition effect
# at fixed cell type.
cond_cols = [c for c in cov.columns if c.startswith("condition_")]
cov_a = cov.copy()
cov_a[cond_cols] = 0.0
cov_b = cov_a.copy()
cov_a[CTRL], cov_b[STIM] = 1.0, 1.0

sign = unit_sign_structure(model, min_genes=10)
assert sign["encoder_frac_negative"] == 0.0, (
    f"{sign['encoder_frac_negative']:.4f} of encoder live weights are negative; "
    "nonneg_encoder is not doing what it claims")
assert sign["dec_out_frac_negative"] == 0.0, (
    f"{sign['dec_out_frac_negative']:.4f} of dec_out live weights are negative; "
    "nonneg_decoder is not doing what it claims and no unit sign below means "
    "anything")

# Check 4. The weight read-out against the model's own pre-activation, batched
# over the same cells so the two sides are never different cell sets -- an
# earlier version compared 512 cells with all of them and called 1.9e-01 a pass.
model.eval()
with torch.no_grad():
    pre_a, pre_b, h_a, h_b, sat = [], [], [], [], []
    for s in range(0, len(x_trans), 4096):
        xb = torch.tensor(x_trans.iloc[s:s + 4096].values, dtype=torch.float32,
                          device=DEV)
        ab = torch.tensor(cov_a.iloc[s:s + 4096].values, dtype=torch.float32,
                          device=DEV)
        bb = torch.tensor(cov_b.iloc[s:s + 4096].values, dtype=torch.float32,
                          device=DEV)
        za = model.encode(xb, ab)[0]
        zb = model.encode(xb, bb)[0]
        # float64 before the tanh, not after. `tanh(pre + d) - tanh(pre)` for a
        # small `d` is a catastrophic cancellation: in float32 it underflows to
        # exactly 0 for any unit with |pre| past about 4, which silently turns a
        # unit's direction into no direction. The cast costs a few seconds and is
        # what makes the sign assertion below meaningful.
        pa = (model.decoder.dec_latent(za) + model.decoder.cov_path(ab)).double()
        pb = (model.decoder.dec_latent(zb) + model.decoder.cov_path(bb)).double()
        pre_a.append(pa.cpu()); pre_b.append(pb.cpu())
        h_a.append(torch.tanh(pa).cpu()); h_b.append(torch.tanh(pb).cpu())
        sat.append((torch.tanh(pa).abs() > 0.99).double().cpu())
PRE_A = torch.cat(pre_a).numpy()
delta_pre_cells = (torch.cat(pre_b) - torch.cat(pre_a)).numpy()
delta_post_cells = (torch.cat(h_b) - torch.cat(h_a)).numpy()
saturated = torch.cat(sat).numpy().mean(axis=0)

direct = pd.Series(model.pathway_contrast(cov_a.to_numpy(), cov_b.to_numpy())
                   .detach().cpu().numpy(), index=pathway_names)
forward_dev = float(np.abs(direct.to_numpy() - delta_pre_cells.mean(0)).max())
cell_spread = float(delta_pre_cells.std(0).max())
assert forward_dev < 1e-4, (
    f"pathway_contrast differs from the forward pass by {forward_dev:.2e}")
assert cell_spread < 1e-4, (
    f"the pre-activation shift varies between cells by {cell_spread:.2e}, so "
    "this was not a clean two-level covariate contrast")

print(f"encoder live weights negative   {sign['encoder_frac_negative']:.4f}"
      "   (must be 0)")
print(f"dec_out live weights negative   {sign['dec_out_frac_negative']:.4f}"
      "   (must be 0; the shipped fit was 0.50)")
print(f"pathway_contrast vs forward     {forward_dev:.2e}"
      "   (float32 precision)")
print(f"shift spread across cells       {cell_spread:.2e}"
      "   (cell-independent by construction)")
sat_half = int((saturated > 0.5).sum())
print(f"saturation, max over units      {saturated.max():.4f}")
print(f"units saturated in >half cells  {sat_half:,} of {len(pathway_names):,}"
      f" ({sat_half / len(pathway_names):.0%})")

## 5. The read-out: one signed number per unit

`pathway_contrast(control, stimulated)` returns
`cov_path.weight @ (cov_stim - cov_ctrl)`, the shift the condition imposes on
each unit's pre-activation. **No cell is encoded and no cell is re-weighted**,
so a change in which cell types were sampled cannot produce a contrast.

Two columns, and they differ in a way worth keeping straight:

- **`delta_pre`** is that exact weight difference — the contrast itself.
- **`member_shift_z`** is what reaches the genes: `delta_post * w_sum /
  sqrt(n_genes)`, where `delta_post` is the change in `tanh(pre)` and `w_sum`
  the unit's total decoder weight. The `1/sqrt(n)` corrects a size tilt that
  otherwise hands the top of the ranking to the smallest units (Spearman
  between |statistic| and set size falls from −0.49 to −0.09 under it).

Under this architecture the two **always agree in sign**, which the next cell
asserts: `tanh` is monotone so `delta_post` carries `delta_pre`'s sign, and
`softplus` makes `w_sum` positive. The ranking is by `member_shift_z`, the same
statistic `01_architecture.py` scored every arm on.

### What saturation costs, and why it is printed above

`tanh` saturates, and this architecture saturates far harder than the one it
replaces. Measured in `01_architecture.py` on the same cells by the same code:

| fit | units saturated in >half their cells | median \|`delta_post`\| |
| --- | ---: | ---: |
| Kang, shipped | 23% | 2.7e-01 |
| **Kang, selected** | **71%** | **5.4e-02** |
| `Zavidij_2020`, shipped | 22% | 6.1e-02 |
| **`Zavidij_2020`, selected** | **80%** | **2.0e-03** |

A pinned unit writes nothing whatever its `delta_pre` is, so `member_shift_z`
is in effect a ranking over the units that still have headroom.

That is why it beats ranking on the weights alone - here the raw weight puts
the target at 284 and `member_shift_z` puts it at 25 - and it is equally the
honest caveat on it: **a unit can be biologically involved and invisible here
because it is pinned.** The count is in `summary.json` so the caveat travels
with the numbers.

What is *not* a cost: a unit writing exactly nothing. That used to happen to
hundreds of units and was arithmetic, not saturation -- `tanh(pre + d) -
tanh(pre)` is a catastrophic cancellation and rounded to zero in float32 past
|pre| ~ 4. Taken in float64 the count is 0, and the cell above asserts it.

### Nine units that can never answer

Reactome has 1,615 sets; this dataset measures 7,573 genes; and **9 of those
sets share no measured gene with it at all**. Those units still exist in the
model -- the latent layer writes to their pre-activation like any other -- but
`dec_out` has no live weight for them, so whatever they do reaches nothing.
They are counted and reported, never silently dropped and never folded into
the saturation count, which is a different failure: a saturated unit has a
direction and cannot express it, an empty one never had one.

### Why no count appears anywhere below

Reactome C2:CP nests strictly — `INTERFERON_ALPHA_BETA_SIGNALING` sits inside
`INTERFERON_SIGNALING` sits inside `CYTOKINE_SIGNALING_IN_IMMUNE_SYSTEM` — so
"how many units responded" counts one family many times and an overlap p-value
over units is not a test of anything. This pipeline reports **signed per-unit
contrasts and ranks**, never a count and never a competitive enrichment
statistic over units.

In [ ]:
w_out = model.decoder.dec_out.effective_weight().detach().double().cpu().numpy()
members = model.decoder.dec_out.mask.detach().double().cpu().numpy()
n_members = members.sum(0)
w_sum = (w_out * members).sum(0)
delta_post = delta_post_cells.mean(0)

units = pd.DataFrame({
    "n_genes": n_members.astype(int),
    "delta_pre": direct.to_numpy(),
    "delta_post": delta_post,
    "member_shift_z": delta_post * w_sum / np.sqrt(np.maximum(n_members, 1)),
    "saturated_frac": saturated,
    "activation_mean": np.tanh(PRE_A).mean(0),
}, index=pathway_names)
units.index.name = "unit"

# The invariant is that no unit *disagrees*, not that none is zero. `tanh` is
# monotone and `softplus` makes `w_sum` positive, so a strict sign flip between
# what a unit does and what it writes is impossible under this architecture and
# would mean the fit is not what it claims. A unit can still land on exactly
# zero: `tanh` saturates, and for a saturated unit no shift reaches the genes at
# all. Those are dead units, counted here rather than hidden.
disagree = int((np.sign(units["delta_pre"])
                * np.sign(units["member_shift_z"]) < 0).sum())
# Two different kinds of silence, which must not be added together. A unit with
# no member genes in this dataset's panel never had a direction to lose -- 9 of
# Kang's 1,615 Reactome sets share no measured gene at all. A unit with members
# but zero written shift is saturated: it has a direction and cannot express it.
n_empty = int((units["n_genes"] == 0).sum())
n_dead = int(((units["member_shift_z"] == 0) & (units["n_genes"] > 0)).sum())
assert disagree == 0, (
    f"{disagree} units write to their genes in the direction opposite to their "
    "own shift; under nonneg_decoder that is impossible, so the fit is not what "
    "it claims to be")
print(f"units whose sign disagrees with their genes   {disagree}   (must be 0)")
print(f"units with no measured member gene            {n_empty} of "
      f"{len(units):,}   (nothing to say, by construction)")
print(f"units saturated so hard nothing reaches genes {n_dead} of "
      f"{len(units):,}")

ranked = units.sort_values("member_shift_z", ascending=False)
units["rank"] = ranked.reset_index().reset_index().set_index("unit")["index"] + 1
units.to_csv(OUT_DIR / "unit_contrasts.csv")

rank_target = int(units.loc[TARGET, "rank"])
rank_parent = int(units.loc[PARENT, "rank"]) if PARENT in units.index else None

print(f"control -> stimulated, {len(units):,} units, "
      f"{len(cov):,} cells held at their own cell type\n")
print(ranked.head(12)[["n_genes", "delta_pre", "member_shift_z"]]
      .rename(index=lambda s: s.replace("REACTOME_", "")[:54]).to_string())
print(f"\n{'TARGET':<8} {TARGET.replace('REACTOME_', '')}  rank {rank_target} "
      f"of {len(units):,}")
print(f"{'parent':<8} {PARENT.replace('REACTOME_', '')}  rank {rank_parent}")

### The same number against the shuffled annotation

A model that beats a shuffle only because masking helps is not using the
biology in the map. `01_architecture.py` fits the identical architecture on a
**row-permuted** adjacency — every pathway keeps its size, its name and the
entire pathway-by-pathway overlap matrix; only which named genes sit in a unit
changes. That arm's rank for the same pathway is read here rather than
recomputed, so the two notebooks cannot disagree.

In [ ]:
arch_ranks = {}
if ARCH_SUMMARY.exists():
    arms = json.loads(ARCH_SUMMARY.read_text())["arms"]
    for arm in ("real_shipped", "perm_shipped", "real_signed", "perm_signed"):
        if arm in arms:
            arch_ranks[arm] = {
                "unit_rank": arms[arm].get("unit_rank_target"),
                "gene_rank": arms[arm].get("gene_rank_target"),
                "best_val": arms[arm].get("best_val")}
    print(f"{'arm':<14}{'unit rank':>11}{'gene-level rank':>18}{'val':>10}")
    for arm, r in arch_ranks.items():
        print(f"{arm:<14}{str(r['unit_rank']):>11}{str(r['gene_rank']):>18}"
              f"{r['best_val']:>10.1f}")
    print("\nthe gene-level column is the old headline read-out: rank 1 whether "
          "or not the\nannotation is real, which is why it is evidence about "
          "the read-out and not a result.")
else:
    print(f"{ARCH_SUMMARY} not found -- run `snakemake architecture` first for "
          "the permuted comparison.")

## 6. Inside the unit: which genes move it

A within-unit question, and the only place gradients appear. Integrated
gradients attributes the **encoder** unit's activation to each input gene: what
the unit reads the pathway through. That is sensitivity, not differential
expression, and the two routinely name different genes — a unit can read a
broad response through a handful of members.

In [ ]:
pidx = pathway_names.index(TARGET)
stim_rows = np.where(cov[STIM].to_numpy() > 0.5)[0][:512]
attr = integrated_gradients(
    model,
    x=torch.tensor(x_trans.iloc[stim_rows].values, dtype=torch.float32, device=DEV),
    pathway_idx=pidx,
    cov=torch.tensor(cov.iloc[stim_rows].values, dtype=torch.float32, device=DEV),
    steps=50)
ga = np.abs(attr.detach().cpu().numpy()).mean(axis=0)
ig = (pd.Series(ga, index=gene_names).sort_values(ascending=False)
      .rename("attribution").to_frame())
ig["share"] = ig["attribution"] / ig["attribution"].sum()
ig["is_member"] = [g in set(adj_df.index[adj_df[TARGET].values > 0])
                   for g in ig.index]
ig.index.name = "gene"
ig.head(40).to_csv(OUT_DIR / "ig_target.csv")
print(f"{TARGET.replace('REACTOME_', '')}: {int(adj_df[TARGET].sum())} member "
      f"genes, top-10 attribution share {ig['share'].head(10).sum():.1%}")
print(ig.head(10).to_string())

## 7. The per-cell table the figures are drawn from

The figures themselves are in `05_figures.py`, a CPU script that reads the CSVs
this notebook writes. The split is deliberate: this notebook fits a model and
needs a GPU for five minutes, while figure work is iterative and belongs on a
laptop. It also means each figure is drawn in exactly one place.

What the script cannot recover from `unit_contrasts.csv` is the target unit's
value in each individual cell, so this section writes it out, with the standard
scanpy coordinates beside it.

**Pre-activation, not activation.** `tanh(pre)` is pinned at 1.000 for most of
these cells, and a plot of it is a flat line at the top of the axis that hides
exactly the thing worth seeing. Both columns are written; the figures use `pre`.

**Every cell is held at `control` in the covariate**, so the gap between the
two conditions in this table is *not* the contrast. It is what the **encoder**
reads off each cell's own expression: `cov_in_encoder=False` means `z` depends
on the transcriptome alone, so a stimulated cell lands higher on this unit
because its genes say so, with nothing told to the model. The covariate's shift
is the single number in `delta_pre`, added on top of every cell equally. The
two routes are separate, and that they agree in direction is the check worth
taking from this table.

The coordinates are standard scanpy from raw counts, written here so the
figures need neither the dataset nor scanpy. No quantitative statement in this
notebook uses them.


In [ ]:
pre_t = PRE_A[:, pidx]
cells = pd.DataFrame({
    "pre": pre_t,
    "activation": np.tanh(pre_t),
    "condition": adata.obs["condition"].astype(str).to_numpy(),
    "cell_type": adata.obs["cell_type"].astype(str).to_numpy(),
})
# Written twice on purpose. The per-cell values are the part no GPU-free run
# can rebuild, so they go to disk before the embedding, which is the step that
# can fail on a small or degenerate panel.
cells.to_csv(OUT_DIR / "target_cells.csv", index=False)

emb = sc.AnnData(counts_df.to_numpy().copy())
emb.obs = adata.obs[["condition", "cell_type"]].copy().reset_index(drop=True)
sc.pp.normalize_total(emb, target_sum=TARGET_SUM)
sc.pp.log1p(emb)
sc.pp.highly_variable_genes(emb, n_top_genes=min(2000, emb.n_vars), flavor="seurat")
emb = emb[:, emb.var["highly_variable"]].copy()
sc.pp.scale(emb, max_value=10)
sc.tl.pca(emb, n_comps=min(50, emb.n_obs - 1, emb.n_vars - 1), random_state=SEED)
sc.pp.neighbors(emb, random_state=SEED)
sc.tl.umap(emb, random_state=SEED)
U = np.asarray(emb.obsm["X_umap"])
cells["umap1"], cells["umap2"] = U[:, 0], U[:, 1]
cells.to_csv(OUT_DIR / "target_cells.csv", index=False)

print(f"{len(cells):,} cells -> {OUT_DIR / 'target_cells.csv'}")
print(f"saturated (|tanh| > 0.99) {float((cells['activation'].abs() > 0.99).mean()):.1%} "
      f"of cells on this unit\n")
print(cells.groupby(["cell_type", "condition"])["pre"].median().unstack()
      .sort_values("stimulated", ascending=False).to_string(
          float_format=lambda v: f"{v:+.3f}"))
print("\nthe four figures: python 05_figures.py   (CPU, no model, seconds)")


## 8. What this run establishes

`summary.json` is the machine-readable version, and `verify_findings.py` reads
it. Read it in this order:

1. **`dec_out_frac_negative` and `encoder_frac_negative` — both zero?** If not,
   no sign below means anything and the fit should be discarded.
2. **`pathway_contrast_vs_forward` — at float32 precision?** The read-out and
   the model must be describing the same thing.
3. **`unit_rank_target`** — where the known answer sits among all units. Under
   the shipped architecture it was last of 1,615.
4. **`perm_unit_rank_target`**, from `01_architecture.py` — where it sits when
   the annotation is shuffled. The gap between 3 and 4 is what the biology in
   the map is worth; a model that scores the same on a shuffle is not using it.

What this does **not** establish: that this architecture fits better. It does
not — it costs validation loss where a dense per-gene term is well estimated,
and `FINDINGS.md` leads with that cost. What it buys is that the contrast is
one signed number per annotated unit, and that the number moves when the
annotation is destroyed.

In [ ]:
summary = {
    "dataset": "kang_2018", "cells": int(adata.n_obs),
    "genes": len(gene_names), "units": len(pathway_names),
    "cell_types": int(adata.obs["cell_type"].nunique()),
    "n_cov": int(n_cov), "epochs": int(TRAIN["epochs"]), "seed": SEED,
    "smoke": SMOKE,
    "architecture": {k: v for k, v in ARCH.items()},
    "train_minutes": round(train_min, 2),
    "best_val_loss": round(best_val, 3),
    "encoder_frac_negative": sign["encoder_frac_negative"],
    "dec_out_frac_negative": sign["dec_out_frac_negative"],
    "pathway_contrast_vs_forward": forward_dev,
    "shift_spread_across_cells": cell_spread,
    "saturation_max": round(float(saturated.max()), 4),
    "units_sign_disagree": disagree,
    "units_dead": n_dead,
    "units_without_members": n_empty,
    "units_saturated_gt_half": sat_half,
    "delta_post_abs_median": round(float(units["delta_post"].abs().median()), 6),
    "target": TARGET,
    "unit_rank_target": rank_target,
    "unit_rank_parent": rank_parent,
    "delta_pre_target": round(float(units.loc[TARGET, "delta_pre"]), 4),
    "member_shift_z_target": round(float(units.loc[TARGET, "member_shift_z"]), 4),
    "top_unit": ranked.index[0],
    "top_10_units": [s.replace("REACTOME_", "") for s in ranked.index[:10]],
    "ig_top_genes": ig.index[:10].tolist(),
    "ig_top10_share": round(float(ig["share"].head(10).sum()), 4),
    "architecture_arms": arch_ranks,
    "perm_unit_rank_target": arch_ranks.get("perm_signed", {}).get("unit_rank"),
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))